# 06 - Services integration (YOLOX + BreastSegmentation)

Este notebook usa `api_stable.MammographyDicom` como pipeline de imagen y prueba inferencia remota contra `yolox-api` y `maseg-api`.

In [ ]:
import os
from pathlib import Path
import base64

import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests

from api_stable.mammography import MammographyDicom

In [ ]:
# Ajusta esta ruta a un DICOM real de tu dataset montado en el contenedor.
DICOM_PATH = Path('/workspace/Data/INbreast/AllDICOMs/20586908')

if not DICOM_PATH.exists():
    raise FileNotFoundError(f'No existe DICOM_PATH: {DICOM_PATH}')

mammo = MammographyDicom.from_dicom(DICOM_PATH).initialize_image().normalize()
arr = mammo.image.to_numpy().astype(np.float32)

arr_min, arr_max = float(arr.min()), float(arr.max())
if arr_max > arr_min:
    arr = (arr - arr_min) / (arr_max - arr_min)
image_uint8 = (arr * 255.0).clip(0, 255).astype(np.uint8)

print('DICOM cargado:', DICOM_PATH)
print('Shape:', image_uint8.shape, 'dtype:', image_uint8.dtype)
print('History:', mammo.image.get_history())

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(image_uint8, cmap='gray')
ax.set_title('Entrada preprocesada con MammographyDicom')
ax.axis('off')
plt.tight_layout()

In [ ]:
in_docker = Path('/.dockerenv').exists()
default_yolox = 'http://yolox-api:8001' if in_docker else 'http://localhost:8001'
default_maseg = 'http://maseg-api:8002' if in_docker else 'http://localhost:8002'

YOLOX_API_BASE = os.getenv('YOLOX_API_BASE', default_yolox)
MASEG_API_BASE = os.getenv('MASEG_API_BASE', default_maseg)

print('Kernel en docker:', in_docker)
print('YOLOX_API_BASE:', YOLOX_API_BASE)
print('MASEG_API_BASE:', MASEG_API_BASE)

for name, base in [('YOLOX', YOLOX_API_BASE), ('MASEG', MASEG_API_BASE)]:
    r = requests.get(f'{base}/health', timeout=30)
    print(name, 'health ->', r.status_code, r.text)

In [ ]:
ok, encoded_png = cv2.imencode('.png', image_uint8)
if not ok:
    raise RuntimeError('No se pudo codificar image_uint8 a PNG')
payload = encoded_png.tobytes()

resp_yolo = requests.post(
    f'{YOLOX_API_BASE}/infer',
    files={'file': ('image.png', payload, 'image/png')},
    data={'input_format': 'BGR'},
    timeout=120,
)
print('YOLOX status:', resp_yolo.status_code)
if resp_yolo.ok:
    yolo_json = resp_yolo.json()
    print('YOLOX detecciones:', yolo_json.get('num_detections'))
    print('Primera deteccion:', (yolo_json.get('detections') or [None])[0])
else:
    print(resp_yolo.text)

In [ ]:
resp_maseg = requests.post(
    f'{MASEG_API_BASE}/infer',
    files={'file': ('image.png', payload, 'image/png')},
    data={'include_pectoral_mask': True, 'fill_holes_in_breast': True},
    timeout=120,
)
print('MAseg status:', resp_maseg.status_code)
if resp_maseg.ok:
    maseg_json = resp_maseg.json()
    print('MAseg summary:', maseg_json.get('summary'))

    b64_png = maseg_json.get('pectoral_mask_png_base64')
    if b64_png:
        mask_png = base64.b64decode(b64_png)
        mask_arr = np.frombuffer(mask_png, dtype=np.uint8)
        mask = cv2.imdecode(mask_arr, cv2.IMREAD_GRAYSCALE)

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(image_uint8, cmap='gray')
        axes[0].set_title('Entrada')
        axes[0].axis('off')

        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title('Mascara pectoral (API)')
        axes[1].axis('off')
        plt.tight_layout()
else:
    print(resp_maseg.text)